In [23]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from statsmodels.tsa.seasonal import STL

input_path = "../dataset/quantity_weekly_decor_outlier_fixed.csv"
output_dir = "../decomposition_stl"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

df = pd.read_csv(input_path)
weeks = df.columns[1:]
dates = pd.date_range(start='2010-12-01', periods=len(weeks), freq='W')
p = 6

for index, row in df.iterrows():
    stock_code = str(row['StockCode'])
    series_data = row[1:].values.astype(float)
    ts = pd.Series(series_data, index=dates)
    
    try:
        stl = STL(ts, period=p, robust=True)
        result = stl.fit()
        
        fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(10, 12), sharex=True)
        axes = [ax1, ax2, ax3, ax4]
        
        ax1.plot(result.observed, color='blue', linewidth=1.5)
        ax1.set_title(f'{stock_code} - Phân rã STL')
        
        ax2.plot(result.trend, color='blue', linewidth=1.5)
        ax2.set_title('Xu hướng')
        
        ax3.plot(result.seasonal, color='blue', linewidth=1.5)
        ax3.set_title('Mùa vụ')
        
        ax4.plot(result.resid, color='blue', linewidth=1.5)
        ax4.set_title('Nhiễu')
        ax4.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.7)
        
        for ax in axes:
            ax.grid(True, linestyle=':', alpha=0.6)
            ax.tick_params(axis='both', which='major', labelsize=9)

        plt.tight_layout()
        plt.savefig(f"{output_dir}/{stock_code}.png", dpi=150)
        plt.close(fig)
        
    except Exception as e:
        print(f"{stock_code}: {e}")

In [24]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import STL

input_path = "../dataset/quantity_weekly_decor_outlier_fixed.csv"

df = pd.read_csv(input_path)
weeks = df.columns[1:]
dates = pd.date_range(start='2010-12-01', periods=len(weeks), freq='W')
p = 6

def analyze_series_stl(ts, period=6):
    res = {}
    
    adf_result = adfuller(ts)
    res['ADF_p_value'] = round(adf_result[1], 4)
    res['Dừng?'] = "Yes" if adf_result[1] < 0.05 else "No"
    
    stl = STL(ts, period=period, robust=True)
    decomp = stl.fit()
    
    resid = decomp.resid
    trend = decomp.trend
    seasonal = decomp.seasonal
    
    var_resid = np.var(resid)
    
    # 3. Độ mạnh xu hướng (Trend Strength)
    # Công thức: Ft = max(0, 1 - Var(Resid) / Var(Trend + Resid))
    var_trend_resid = np.var(trend + resid)
    if var_trend_resid == 0:
        trend_strength = 0
    else:
        trend_strength = max(0, 1 - var_resid / var_trend_resid)
    
    res['Độ mạnh xu hướng'] = round(trend_strength, 4)
    res['Có xu hướng?'] = "Yes" if trend_strength > 0.6 else "No"
    
    # 4. Độ mạnh mùa vụ (Seasonal Strength)
    # Công thức: Fs = max(0, 1 - Var(Resid) / Var(Seasonal + Resid))
    var_seasonal_resid = np.var(seasonal + resid)
    if var_seasonal_resid == 0:
        seasonal_strength = 0
    else:
        seasonal_strength = max(0, 1 - var_resid / var_seasonal_resid)
        
    res['Độ mạnh mùa vụ'] = round(seasonal_strength, 4)
    res['Có mùa vụ?'] = "Yes" if seasonal_strength > 0.6 else "No"
    
    return res

final_results = []

for index, row in df.iterrows():
    stock_code = str(row['StockCode'])
    series_data = pd.to_numeric(row[1:], errors='coerce').values.astype(float)
    
    series_data = np.nan_to_num(series_data, nan=1e-6)
    series_data[series_data <= 0] = 1e-6
    ts = pd.Series(series_data, index=dates)
    
    try:
        stats = analyze_series_stl(ts, period=p)
        stats['StockCode'] = stock_code
        final_results.append(stats)
    except Exception as e:
        print(f"{stock_code}: {e}")

df_results = pd.DataFrame(final_results)
cols = ['StockCode', 'Dừng?', 'ADF_p_value', 'Có xu hướng?', 'Độ mạnh xu hướng', 'Có mùa vụ?', 'Độ mạnh mùa vụ']
df_results = df_results[cols]

print(df_results.to_string(index=False))

StockCode Dừng?  ADF_p_value Có xu hướng?  Độ mạnh xu hướng Có mùa vụ?  Độ mạnh mùa vụ
    22469   Yes       0.0000           No            0.5121         No          0.1086
    22961   Yes       0.0350           No            0.4103         No          0.1059
    21181    No       0.2047           No            0.5821         No          0.0000
    21137    No       0.5692          Yes            0.8384         No          0.2730
    84978   Yes       0.0012          Yes            0.9407        Yes          0.6526
    21326    No       0.1840           No            0.2488         No          0.0000
    22969    No       0.0720          Yes            0.6090         No          0.1528
    79321   Yes       0.0482           No            0.5330         No          0.0000
    84945    No       0.0806           No            0.4692         No          0.0000
    22470   Yes       0.0010           No            0.5217         No          0.0218
    21175    No       0.3561          Yes  

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import STL

input_path = "../dataset/quantity_weekly_decor_outlier_fixed.csv"

df = pd.read_csv(input_path)
weeks = df.columns[1:]
dates = pd.date_range(start='2010-12-01', periods=len(weeks), freq='W')
p = 6

def analyze_series_stl(ts, period=6):
    res = {}
    
    # ADF trên chuỗi gốc
    adf_original = adfuller(ts)
    res['ADF_p_value_Ori'] = round(adf_original[1], 4)
    res['Dừng_Ori?'] = "Yes" if adf_original[1] < 0.05 else "No"
    
    # Lấy vi phân bậc 1 và kiểm định ADF lại
    ts_diff = ts.diff().dropna()
    adf_diff = adfuller(ts_diff)
    res['ADF_p_value_Diff'] = round(adf_diff[1], 4)
    res['Dừng_Diff?'] = "Yes" if adf_diff[1] < 0.05 else "No"
    
    stl = STL(ts, period=period, robust=True)
    decomp = stl.fit()
    
    resid = decomp.resid
    trend = decomp.trend
    seasonal = decomp.seasonal
    var_resid = np.var(resid)
    
    var_trend_resid = np.var(trend + resid)
    if var_trend_resid == 0:
        trend_strength = 0
    else:
        trend_strength = max(0, 1 - var_resid / var_trend_resid)
    
    res['Độ mạnh xu hướng'] = round(trend_strength, 4)
    res['Có xu hướng?'] = "Yes" if trend_strength > 0.6 else "No"
    
    var_seasonal_resid = np.var(seasonal + resid)
    if var_seasonal_resid == 0:
        seasonal_strength = 0
    else:
        seasonal_strength = max(0, 1 - var_resid / var_seasonal_resid)
        
    res['Độ mạnh mùa vụ'] = round(seasonal_strength, 4)
    res['Có mùa vụ?'] = "Yes" if seasonal_strength > 0.6 else "No"
    
    return res

final_results = []

for index, row in df.iterrows():
    stock_code = str(row['StockCode'])
    series_data = pd.to_numeric(row[1:], errors='coerce').values.astype(float)
    series_data = np.nan_to_num(series_data, nan=1e-6)
    series_data[series_data <= 0] = 1e-6
    ts = pd.Series(series_data, index=dates)
    
    try:
        stats = analyze_series_stl(ts, period=p)
        stats['StockCode'] = stock_code
        final_results.append(stats)
    except Exception as e:
        print(f"{stock_code}: {e}")

df_results = pd.DataFrame(final_results)
cols = ['StockCode', 'Dừng_Ori?', 'ADF_p_value_Ori', 'Dừng_Diff?', 'ADF_p_value_Diff', 'Có xu hướng?', 'Độ mạnh xu hướng', 'Có mùa vụ?', 'Độ mạnh mùa vụ']
df_results = df_results[cols]

print(df_results.to_string(index=False))

StockCode Dừng_Ori?  ADF_p_value_Ori Dừng_Diff?  ADF_p_value_Diff Có xu hướng?  Độ mạnh xu hướng Có mùa vụ?  Độ mạnh mùa vụ
    22469       Yes           0.0000        Yes            0.0186           No            0.5121         No          0.1086
    22961       Yes           0.0350        Yes            0.0008           No            0.4103         No          0.1059
    21181        No           0.2047        Yes            0.0000           No            0.5821         No          0.0000
    21137        No           0.5692        Yes            0.0000          Yes            0.8384         No          0.2730
    84978       Yes           0.0012        Yes            0.0004          Yes            0.9407        Yes          0.6526
    21326        No           0.1840        Yes            0.0000           No            0.2488         No          0.0000
    22969        No           0.0720        Yes            0.0008          Yes            0.6090         No          0.1528
    7932

In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from statsmodels.tsa.seasonal import seasonal_decompose

input_path = "../dataset/quantity_weekly_decor_outlier_fixed.csv"
output_dir = "../decomposition_seasonal"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

df = pd.read_csv(input_path)
weeks = df.columns[1:]
dates = pd.date_range(start='2010-12-01', periods=len(weeks), freq='W')
p = 6

for index, row in df.iterrows():
    stock_code = str(row['StockCode'])
    series_data = row[1:].values.astype(float)
    series_data[series_data <= 0] = 1e-6
    ts = pd.Series(series_data, index=dates)
    
    try:
        res_add = seasonal_decompose(ts, model='additive', period=p)
        res_mul = seasonal_decompose(ts, model='multiplicative', period=p)
        
        fig, axes = plt.subplots(4, 2, figsize=(16, 12), sharex='col')
        
        cols = [res_add, res_mul]
        titles = ['Phân rã cộng (Additive)', 'Phân rã nhân (Multiplicative)']
        
        for i, res in enumerate(cols):
            axes[0, i].plot(res.observed, color='blue', linewidth=1.2)
            axes[0, i].set_title(f'{stock_code} - {titles[i]}')
            
            axes[1, i].plot(res.trend, color='blue', linewidth=1.2)
            axes[1, i].set_title('Xu hướng')
            
            axes[2, i].plot(res.seasonal, color='blue', linewidth=1.2)
            axes[2, i].set_title('Mùa vụ')
            
            axes[3, i].plot(res.resid, color='blue', linewidth=1.2)
            axes[3, i].set_title('Nhiễu')
            
            ref_line = 0 if i == 0 else 1
            axes[3, i].axhline(ref_line, color='red', linestyle='--', linewidth=1, alpha=0.5)

        for ax in axes.flat:
            ax.grid(True, linestyle=':', alpha=0.6)
            ax.tick_params(axis='both', which='major')

        plt.tight_layout()
        plt.savefig(f"{output_dir}/{stock_code}.png", dpi=150)
        plt.close(fig)
        
    except Exception as e:
        print(f"{stock_code}: {e}")